# 46. The rescaling control, on all fifteen pairs

This trains nothing. Every number comes from a forward pass.

## The question

When a task is unlearned, two things happen at once. The generated weights get
smaller, and the representation changes. Both can move a rank measure.

Both rank measures divide the singular values by their own sum, so multiplying
the whole feature matrix by a constant cannot change them. But the path from
the generator's weights to those features is not a simple multiplication. It
runs through ReLU and through batch normalisation with fixed statistics, so
shrinking the weights pushes units across their firing threshold and can
reshape the spectrum by itself.

So: how much of the rank change at the unlearn is just the weights being
smaller, and how much is the representation?

## How it is answered

Four readings per pair, all forward passes, nothing trained:

| Arm | What it is |
| --- | --- |
| `b9` | the model at the end of the stream, as it is |
| `b10` | the model after the unlearn, as it is |
| `b10_at_b9_norm` | the post-unlearn weights multiplied by one constant, so their norm matches b9 |
| `b9_at_b10_norm` | the end-of-stream weights multiplied by the constant that imposes b10's norm |

If `b10_at_b9_norm` falls back to where `b9` was, the weight scale explains the
whole change and the unlearn result is an artifact. If it stays where `b10`
was, the scale explains none of it. Anything between is a partial explanation
and gets reported as such.

## Why fifteen pairs and not one

This control was run once before, on 8 November, on a single pair from the old
study. It found the norm carried about two thirds of the effect.

One pair cannot say whether two thirds is the real figure or that pair's luck.
Job 2 and job 3 left fifteen before-and-after pairs, five seeds in each of
three orders, so the share can be reported with a spread.

That matters more than usual here. Reading the five order_A records showed the
rank measures move almost as much between seeds as nine tasks of learning move
them. A control on one pair of a noisy measure is not a control.

## Cost

Each pair means downloading two model files of about 539 MB. Fifteen pairs is
about 16.2 GB in total, but only one pair is on disk at a time: each is
deleted after it is read. The compute is 60 forward passes over 256 images.

A GPU runtime makes this minutes. It will run on CPU but will take hours.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, shutil, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-rescale")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

import torch
from uncle import checkpoint
from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import HyperNetwork, build_target
from uncle.plasticity_indicators import (fixed_image_sets, flatten,
                                         reference_code, srank)
from uncle.probe import _features_and_logits, effective_rank
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.trainer import UnCLe

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  ", torch.cuda.get_device_name(0))
else:
    print("   No GPU. This will work but will take hours instead of minutes.")

## 2. Settings

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# The fifteen runs: three orders, five seeds each. Each one saved a model at
# boundary 9, the end of the nine-task stream, and at boundary 10, after its
# task was unlearned. That pair is what this control reads.
ORDERS = {"order_A": "0", "order_B": "4", "order_C": "8"}   # order -> task unlearned
SEEDS = (0, 1, 2, 3, 4)
PAIRS = [(order, seed) for order in ORDERS for seed in SEEDS]

IMAGES_PER_SET = 256
INCLUDE_EUROSAT = True

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
EUROSAT_ROOT = Path("/content/eurosat") if IN_COLAB else REPO / "data/eurosat"
CACHE = Path("/content/rescale") if IN_COLAB else REPO / "outputs/rescaling-control"
CACHE.mkdir(parents=True, exist_ok=True)

# Each model file is about 539 MB and two are needed per pair. Deleting them
# after each pair keeps about 1.1 GB on disk instead of 16.2 GB. Set this to
# False only if you have the space and want to re-read a pair without
# downloading it again.
DELETE_AFTER_EACH_PAIR = True

print(f"{len(PAIRS)} pairs: " + ", ".join(f"{o}/seed{s}" for o, s in PAIRS[:4])
      + f", ... and {len(PAIRS) - 4} more")
print("About", round(len(PAIRS) * 2 * 0.539, 1), "GB to download in total,",
      "roughly 1.1 GB on disk at a time" if DELETE_AFTER_EACH_PAIR else "all kept")

## 3. Connect to R2

In [ ]:
def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip('"').strip("'")
    return value


store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))

# The settings the runs used, read back rather than retyped, so this control
# cannot quietly rebuild the model with a different shape from the one the
# checkpoints were written by.
plan = json.loads(store.download(
    f"uncle/indicators/{STUDY_ID}/job2_summary.json", CACHE).read_text(encoding="utf-8"))
print("Runs used learning rate", plan["learning_rate"],
      "| epochs", plan["epochs"], "| batch", plan["batch_size"],
      "| init", plan["initialization"])

## 4. Build the image sets

In [ ]:
from torch.utils.data import ConcatDataset

# The same pictures every run was measured on. The draw is seeded inside the
# library, so these are the identical images the boundary records used. If
# they were not, the numbers below could not be compared with the records.
STREAM_TASKS = [str(i) for i in range(9)]

tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42,
                    include=tuple(STREAM_TASKS + [PROBE_TASK]), download=True)

sources = {
    "in_stream": ConcatDataset([tasks[name]["test"] for name in STREAM_TASKS]),
    "probe": tasks[PROBE_TASK]["test"],
}
if INCLUDE_EUROSAT:
    from torchvision.datasets import EuroSAT
    from torchvision.transforms import ToTensor
    eurosat = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(), download=True)
    sources["eurosat"] = eurosat

image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)
for name, images in image_sets.items():
    print(f"{name:<10} {tuple(images.shape)}")

## 5. The four arms

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@torch.no_grad()
def features_scaled(hypernet, target, trainer, images, code, scale, batch_size=64):
    """The library's feature read, with every generated weight multiplied by `scale`.

    This is copied from `plasticity_indicators.features` rather than imported
    with an extra argument, because the library version is what produced every
    number already on record, and a one-off control has no business changing
    it. The only difference is the line that multiplies the weights.
    """
    weights = hypernet.weights_from_code(code)
    if scale != 1.0:
        weights = {name: value * scale for name, value in weights.items()}
    modes = hypernet.training, target.training
    hypernet.eval()
    target.eval()
    collected = []
    try:
        for start in range(0, len(images), batch_size):
            batch = images[start:start + batch_size].to(trainer.device)
            captured, _ = _features_and_logits(hypernet, target, trainer, None,
                                               batch, weights=weights,
                                               buffers=trainer.buffer_template)
            collected.append(captured.float().cpu())
    finally:
        hypernet.train(modes[0])
        target.train(modes[1])
    return torch.cat(collected)


def load_model(path, config):
    """A fresh model with one saved checkpoint loaded into it."""
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, None, progress=False)
    payload = torch.load(path, map_location=DEVICE, weights_only=False)
    checkpoint.restore(payload, hypernet=hypernet, uncle=trainer)
    return hypernet, target, trainer


def read_arms(order, seed):
    """Download one pair, read the four arms, return the numbers.

    The two scale factors are the only thing that differs between the arms.
    `up` grows the post-unlearn weights back to the size they were before the
    unlearn. `down` shrinks the intact weights to the size the unlearn left.
    """
    settings = study_defaults(order)
    settings["requests"] = settings["requests"] + (("forget", ORDERS[order]),)
    config = Config(**settings, seed=seed,
                    learning_rate=plan["learning_rate"], epochs=plan["epochs"],
                    batch_size=plan["batch_size"],
                    initialization=plan["initialization"], device=DEVICE)

    base = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}"
    paths = {name: store.download(f"{base}/boundary_{t}.pt", CACHE)
             for name, t in (("b9", 9), ("b10", 10))}

    models, norms = {}, {}
    for name, path in paths.items():
        models[name] = load_model(path, config)
        hypernet = models[name][0]
        norms[name] = float(flatten(hypernet, reference_code(hypernet))["target"].norm())

    up = norms["b9"] / norms["b10"]
    down = norms["b10"] / norms["b9"]
    arms = [("b9", "b9", 1.0), ("b10", "b10", 1.0),
            ("b10_at_b9_norm", "b10", up), ("b9_at_b10_norm", "b9", down)]

    out = {"order": order, "seed": seed,
           "norm_b9": norms["b9"], "norm_b10": norms["b10"],
           "norm_ratio": norms["b10"] / norms["b9"], "arms": {}}
    for label, which, scale in arms:
        hypernet, target, trainer = models[which]
        code_vec = reference_code(hypernet)
        row = {}
        for name, images in image_sets.items():
            matrix = features_scaled(hypernet, target, trainer, images, code_vec, scale)
            row[f"effective_rank_{name}"] = effective_rank(matrix)
            row[f"srank_{name}"] = srank(matrix)
        out["arms"][label] = row

    del models
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if DELETE_AFTER_EACH_PAIR:
        for path in paths.values():
            Path(path).unlink(missing_ok=True)
    return out

## 6. Read all fifteen pairs

In [ ]:
# Re-running this cell skips pairs already read, so an interrupted session
# continues rather than downloading 16 GB again.
try:
    readings
except NameError:
    readings = {}

for index, (order, seed) in enumerate(PAIRS, start=1):
    key = f"{order}_seed{seed}"
    if key in readings:
        continue
    began = time.perf_counter()
    print(f"[{index}/{len(PAIRS)}] {key} ...", end=" ", flush=True)
    readings[key] = read_arms(order, seed)
    row = readings[key]
    raw = (row["arms"]["b10"]["effective_rank_in_stream"]
           - row["arms"]["b9"]["effective_rank_in_stream"])
    left = (row["arms"]["b10_at_b9_norm"]["effective_rank_in_stream"]
            - row["arms"]["b9"]["effective_rank_in_stream"])
    print(f"done in {time.perf_counter() - began:.0f}s   "
          f"norm x{row['norm_ratio']:.3f}   "
          f"rank change {raw:+.3f}, with the norm held fixed {left:+.3f}")

print(f"\n{len(readings)} of {len(PAIRS)} pairs read")

## 7. The answer

In [ ]:
from statistics import mean, stdev

# For one pair and one measure:
#   raw        what the unlearn did to the measure
#   residual   what is left after the weights are put back to their old size
#   share      how much of the raw change the weight size accounts for
#
# A share near 1 means the whole thing was the weights getting smaller. Near 0
# means the weights explain none of it. Shares are only meaningful when the raw
# change is big enough to divide by, so tiny changes are skipped.
def shares(measure, floor=0.05):
    out = []
    for key, row in readings.items():
        b9 = row["arms"]["b9"][measure]
        raw = row["arms"]["b10"][measure] - b9
        residual = row["arms"]["b10_at_b9_norm"][measure] - b9
        if abs(raw) < floor:
            continue
        out.append({"run": key, "raw": raw, "residual": residual,
                    "share": 1 - residual / raw})
    return out


for measure in ("effective_rank_in_stream", "effective_rank_probe",
                "effective_rank_eurosat", "srank_in_stream"):
    rows = shares(measure)
    if not rows:
        print(f"{measure}: every change was too small to divide by")
        continue
    raw_mean = mean(r["raw"] for r in rows)
    res_mean = mean(r["residual"] for r in rows)
    same_raw = sum(r["raw"] > 0 for r in rows)
    same_res = sum(r["residual"] > 0 for r in rows)
    # Two ways to express the share, and they are not interchangeable.
    #
    # The ratio of the averages is the one to report. Averaging the per-run
    # ratios instead divides by each run's own raw change, and some of those
    # are near zero, which sends a single ratio to -78% or +124% and drags the
    # average with it. The per-run numbers print below as a diagnostic only.
    stable = (1 - res_mean / raw_mean) * 100 if raw_mean else float("nan")
    per_run = [r["share"] * 100 for r in rows]
    print()
    print(f"{measure}   ({len(rows)} of {len(readings)} pairs had a change "
          f"big enough to read)")
    print(f"  the unlearn moved it by      {raw_mean:+.3f} on average, "
          f"same direction in {same_raw} of {len(rows)} runs")
    print(f"  with the weight size held    {res_mean:+.3f} on average, "
          f"same direction in {same_res} of {len(rows)}")
    print(f"  so the weight size explains  {stable:.0f}% of it")
    if len(per_run) > 1:
        print(f"  (averaging the per-run ratios gives {mean(per_run):.0f}% with "
              f"a spread of {stdev(per_run):.0f} points. That statistic divides "
              f"by each run's own change, which is near zero in some runs, so "
              f"the line above uses the ratio of the averages instead.)")

print()
print("=" * 78)
print("Per run, for effective rank on the in-stream images")
print("=" * 78)
print(f"{'run':>16}{'norm ratio':>12}{'raw change':>12}{'left over':>11}{'explained':>11}")
for row in shares("effective_rank_in_stream"):
    key = row["run"]
    print(f"{key:>16}{readings[key]['norm_ratio']:>12.3f}{row['raw']:>12.3f}"
          f"{row['residual']:>11.3f}{row['share'] * 100:>10.0f}%")

summary = {"study": STUDY_ID, "code_revision": HEAD,
           "pairs": len(readings), "readings": readings,
           "shares": {m: shares(m) for m in
                      ("effective_rank_in_stream", "effective_rank_probe",
                       "effective_rank_eurosat", "srank_in_stream")}}
path = CACHE / "rescaling_control.json"
path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
store.upload(path, f"uncle/indicators/{STUDY_ID}/rescaling_control.json",
             overwrite=True)
print()
print("Written:", path, "and uploaded to R2.")
print("Share that file and the tables above.")

## 8. How to read the result

The 8 November run of this control, on one pair of the old study, found the
weight size carried about two thirds of the rank change at the unlearn.

If the fifteen pairs agree, that figure holds and the conclusion from it
stands: a measure that reports improvement because the damaging step happened
to shrink the weights is not reporting damage.

If the share varies widely between pairs, then two thirds was that pair's
luck, in the same way the task-order effect turned out to be. The spread
printed above is what tells them apart.

Either way, what is left after the weight size is held fixed is the part that
is genuinely about the representation, and that is the number section 4.3
should quote.